# Model Setup and Pathway Insertion

In this notebook, we are going to load the genome scale model of wild type *E. coli* (`iJO1366`).
Our goal is to introduce a new chemical pathway so the bacteria can produce Naringenin.
First, let's load the digital map of the bacteria and see how big it is.

In [1]:
import cobra
from cobra import Metabolite, Reaction

model = cobra.io.load_model("iJO1366")   #loading the E.coli model from the database


print(f"Original model has {len(model.reactions)} reactions.")   #print out how many reactions are currently in the cell.    

Original model has 2583 reactions.


### Defining the first plant enzyme (TAL)

*E. coli* naturally produces an amino acid called **L-Tyrosine**. To start our pathway, we need to convert this L-Tyrosine into a new chemical called **p-Coumaric acid**.

To do this, we will introduce the plant enzyme **Tyrosine ammonia-lyase (TAL)**.
Because *E. coli* has never seen p-Coumaric acid before, we must first define it as a new metabolite. Then, we will define the TAL reaction and specify its stoichiometry (what it consumes and what is produces).

In [2]:
#Creating new plant metabolite: p-Coumaric acid
#'c' means it exists in the cytosol (inside the cell)
p_coumaric_acid = Metabolite(
    'pcoum_c',
    name = 'p-Coumaric acid',
    compartment = 'c'
)

#Initializing the TAL enzyme (Reaction)
rxn_tal = Reaction('TAL')
rxn_tal.name = 'Tyrosine ammonia lyase'

#Setting directionality (0 to 1000 => only flows strictly forward)
rxn_tal.lower_bound = 0
rxn_tal.upper_bound = 1000

#Defining the chemistry (Stoichiometry)-
#Negative values are consumed (reactants)
#Positive values are produced (products)
rxn_tal.add_metabolites({
    model.metabolites.get_by_id('tyr__L_c'): -1,     #native L-tyrosine consumed
    p_coumaric_acid: 1,                             #New p-Coumaric acid produced
    model.metabolites.get_by_id('nh4_c'): 1,         #native Ammonia by-product produced
})

#Inserting new enzyme in E.coli model.
model.add_reactions([rxn_tal])

#verifying
print(f"Now model has {len(model.reactions)} reactions.")


Now model has 2584 reactions.


### Completing the pathway

With p-Coumaric acid synthesized, introducing the remaining three enzymes:
1. **4CL (4-Coumarate-CoA ligase):** It activates p-Coumaric acid by attaching a Coenzyme A (CoA) molecule. This step costs cell energy (ATP).
2. **CHS (Chalcone synthase):** It's a bottleneck enzyme, combines the activated p-Coumaroyl-CoA with 3 molecules of Malonyl-CoA to form Naringenin chalcone.
3. **CHI (Chalcone isomerase):** It folds the chalcone into our final product **Naringenin**.

Finally, we must add a "Demand" reaction. In computational modelling, a cell cannot accumulate mass infinitely. The demand reaction act as a drain, simulating the harvesting of Naringenin out of the cell so the mathematical matrix can solve.


In [3]:
#Defining the remaining new metabolites
p_coumaroyl_coa = Metabolite('pcoumcoa_c', name='p-Coumaroyl-CoA', compartment='c')
naringenin_chalcone =  Metabolite('narchal_c', name='Naringenin Chalcone', compartment='c')
naringenin =  Metabolite('naring_c', name='Naringenin', compartment='c')

#Building 4CL enzyme reaction
rxn_4cl = Reaction('4CL')
rxn_4cl.name = '4-Coumarate-CoA ligase'
rxn_4cl.lower_bound, rxn_4cl.upper_bound = 0,1000
rxn_4cl.add_metabolites({
    p_coumaric_acid: -1,                          #consumes our previous product
    model.metabolites.get_by_id('atp_c'): -1,     #consumes native atp
    model.metabolites.get_by_id('coa_c'): -1,     #consumes native CoA
    p_coumaroyl_coa: 1,                           #produces p-Coumaroyl-CoA
    model.metabolites.get_by_id('amp_c'): 1,      #produces AMP by-product
    model.metabolites.get_by_id('ppi_c'): 1,      #produces pyrophosphate by-product
})

#Building CHS enzyme reaction
rxn_chs = Reaction('CHS')
rxn_chs.name = 'Chalcone synthase'
rxn_chs.lower_bound, rxn_chs.upper_bound = 0,1000
rxn_chs.add_metabolites({
    p_coumaroyl_coa: -1,                          
    model.metabolites.get_by_id('malcoa_c'): -3,  #Massive drain: consumes 3 Malonyl-CoA 
    model.metabolites.get_by_id('coa_c'): 4,      #Releases 4 CoA back to the cell
    naringenin_chalcone: 1,                       
    model.metabolites.get_by_id('co2_c'): 3,      #Releases 3 CO2
})

#Building CHI enzyme reaction
rxn_chi = Reaction('CHI')
rxn_chi.name = 'Chalcone isomerase'
rxn_chi.lower_bound, rxn_chs.upper_bound = 0,1000
rxn_chi.add_metabolites({
    naringenin_chalcone : -1,
    naringenin          :  1,
})

#Creating a demand reaction by harvesting the production (for static model).
naring_demand = model.add_boundary(naringenin, type='demand')

#Adding all reactions to the model.
model.add_reactions([rxn_4cl,rxn_chs,rxn_chi])

#Saving the digital strain so we can use it in notebook 2.
cobra.io.save_json_model(model, "../data/iJO1366_naringenin.json")

#Final completion and reactions
print(f"Pathway Completed! Final model has {len(model.reactions)} reactions.")
print("Model saved to the data folder.")

Pathway Completed! Final model has 2588 reactions.
Model saved to the data folder.
